In [1]:
import time
import json
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
import gensim.downloader as api
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

/Users/isaac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from Data.Tool import *
from Model.Tool import *
from Model.model import *
# Pour l'utiliser (remplace 'train.txt' par le nom de ton fichier) :
df_train = parser_donnees_presidents('Data/train/corpus.tache1.learn.utf8')
df_train['Texte_Nettoye'] = df_train['Texte'].apply(nettoyer_texte)
X = df_train['Texte_Nettoye']
y = pd.Series(df_train['Cible_Chirac'])

In [3]:
# --- Configuration ---
MAX_FEATURES = 5000
K_FOLDS = 5
# Initialisation
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
oof_probabilites = np.zeros(len(y))
oof_predictions = np.zeros(len(y))
scores_folds = []

print(f"--- Début de la Validation Croisée à {K_FOLDS} Folds ---")
temps_debut_total = time.time()
durees_folds = []

# 2. Boucle sur chaque Fold
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    start_fold = time.time()
    print(f"Entraînement du Fold {fold + 1}/{K_FOLDS}...")
    
    # Séparation (X est du pandas donc .iloc ok, y est du numpy donc indexation directe)
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Vectorisation
    vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf = vectorizer.transform(X_val)
    
    # Modèle
    modele = LogisticRegression(max_iter=5000, random_state=42)
    modele.fit(X_train_tfidf, y_train)
    
    # Inférence
    probs = modele.predict_proba(X_val_tfidf)[:, 1]
    preds = modele.predict(X_val_tfidf)
    
    # Stockage OOF
    oof_probabilites[val_idx] = probs
    oof_predictions[val_idx] = preds
    
    # Métriques
    metriques = {
        'Fold': fold + 1,
        'Accuracy': accuracy_score(y_val, preds),
        'Precision': precision_score(y_val, preds),
        'Recall': recall_score(y_val, preds),
        'F1_Score': f1_score(y_val, preds),
        'ROC_AUC': roc_auc_score(y_val, probs)
    }
    scores_folds.append(metriques)
    
    durees_folds.append(time.time() - start_fold)

temps_total = time.time() - temps_debut_total
print("--- Fin de l'entraînement ---\n")

# --- 3. Consolidation et Métadonnées ---

# A. Sauvegarde des scores
df_scores_folds = pd.DataFrame(scores_folds).drop(columns=['Fold'])
moyennes_metriques = df_scores_folds.mean().to_dict()
stds_metriques = df_scores_folds.std().to_dict()

# 2. Calcul des statistiques pour le temps
temps_moyen = np.mean(durees_folds)
temps_std = np.std(durees_folds)

# 3. Récupération des paramètres (exemple LogReg)
config_modele = modele.get_params()

# 4. Construction du dictionnaire final standardisé
resultats_complets = {
    "config": {
        "nom_modele": "Logistic_Regression_Baseline",
        "representation": "TF-IDF",
        "nb_parametres_totaux": int((X_train_tfidf.shape[1] * 1) + 1),
        "taille_vocabulaire_entree": MAX_FEATURES,
        "nb_folds": K_FOLDS,
        "epochs_par_fold": config_modele.get('max_iter'),
        "batch_size": "Full-Batch",
        "learning_rate": config_modele.get('C'),
        "device_utilise": "cpu"
    },
    "temps_execution": {
        "total_sec": round(temps_total, 2),
        "moyen_par_fold_sec": round(temps_moyen, 4),
        "std_par_fold_sec": round(temps_std, 4)  # Écart-type du temps ajouté ici
    },
    "performances_moyennes": {
        metric: {
            "mean": round(moyennes_metriques[metric], 4),
            "std": round(stds_metriques[metric], 4)
        } for metric in moyennes_metriques
    },
    "details_par_fold": scores_folds
}

# 5. Sauvegarde
filename = "results_logistic_regression_TF-ID.json"
with open(filename, 'w') as f:
    json.dump(resultats_complets, f, indent=4)

print(f"✅ Rapport complet sauvegardé : {filename}")

# Affichage console pour vérification rapide
print(f"⏱️ Temps : {temps_moyen:.3f}s (± {temps_std:.3f}s)")
print(f"🎯 F1-Score : {resultats_complets['performances_moyennes']['F1_Score']['mean']} (± {resultats_complets['performances_moyennes']['F1_Score']['std']})")

--- Début de la Validation Croisée à 5 Folds ---
Entraînement du Fold 1/5...
Entraînement du Fold 2/5...
Entraînement du Fold 3/5...
Entraînement du Fold 4/5...
Entraînement du Fold 5/5...
--- Fin de l'entraînement ---

✅ Rapport complet sauvegardé : results_logistic_regression_TF-ID.json
⏱️ Temps : 0.471s (± 0.025s)
🎯 F1-Score : 0.9396 (± 0.0015)


In [6]:
# --- Paramètres d'entraînement ---
MAX_FEATURES = 5000
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialisation
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
oof_probs = np.zeros(len(df_train))
oof_preds = np.zeros(len(df_train))
scores_folds = []

print(f"--- Début de la Validation Croisée sur {DEVICE} ---")

temps_debut_total = time.time()
durees_folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🚀 Fold {fold + 1}/{k}")
    # Séparation et Vectorisation
    X_train_raw, X_val_raw = X.iloc[train_idx], X.iloc[val_idx]
    y_train_raw, y_val_raw = y.iloc[train_idx], y.iloc[val_idx]
    
    vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)
    X_train_tfidf = vectorizer.fit_transform(X_train_raw).toarray()
    X_val_tfidf = vectorizer.transform(X_val_raw).toarray()
    
    # Conversion en Tenseurs PyTorch
    X_train_tensor = torch.tensor(X_train_tfidf, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_raw.values, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_tfidf, dtype=torch.float32)
    
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)

    # Initialisation Modèle, Loss et Optimizer
    model = MLP(vocab_size=MAX_FEATURES).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss() # Adapté pour une sortie unique (logit)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- BOUCLE D'ENTRAÎNEMENT ---
    start_fold = time.time()
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        end_fold = time.time()
        durees_folds.append(end_fold - start_fold) 

    temps_total = time.time() - temps_debut_total
    # --- INFÉRENCE (Validation) ---
    model.eval()
    with torch.no_grad():
        X_val_tensor = X_val_tensor.to(DEVICE)
        logits = model(X_val_tensor)
        # Sigmoid pour transformer le logit en probabilité (0 à 1)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

    # Sauvegarde OOF
    oof_probs[val_idx] = probs
    oof_preds[val_idx] = preds
    
    # Métriques
    metriques = {
        'Fold': fold + 1,
        'Accuracy': accuracy_score(y_val_raw, preds),
        'Precision': precision_score(y_val_raw, preds),
        'Recall': recall_score(y_val_raw, preds),
        'F1_Score': f1_score(y_val_raw, preds),
        'ROC_AUC': roc_auc_score(y_val_raw, probs)
    }
    scores_folds.append(metriques)
    print(f"Accuracy: {metriques['Accuracy']:.4f}| F1-Score: {metriques['F1_Score']:.4f} | ROC-AUC: {metriques['ROC_AUC']:.4f}")

# 3. Consolidation finale
df_scores_folds = pd.DataFrame(scores_folds).drop(columns=['Fold'])
moyennes_metriques = df_scores_folds.mean().to_dict()
stds_metriques = df_scores_folds.std().to_dict()

# 2. Calcul des statistiques pour le temps
# durees_folds doit être la liste des temps de chaque fold (time.time() - start_fold)
temps_total = sum(durees_folds)
temps_moyen = np.mean(durees_folds)
temps_std = np.std(durees_folds)

# 3. Construction du dictionnaire final standardisé
resultats_complets = {
    "config": {
        "nom_modele": "MLP_Baseline",
        "representation": "TF-IDF",
        "nb_parametres_totaux": int(count_parameters(model)),
        "taille_vocabulaire_entree": MAX_FEATURES,
        "nb_folds": k, # ou K_FOLDS
        "epochs_par_fold": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "device_utilise": str(DEVICE)
    },
    "temps_execution": {
        "total_sec": round(temps_total, 2),
        "moyen_par_fold_sec": round(temps_moyen, 4),
        "std_par_fold_sec": round(temps_std, 4)
    },
    "performances_moyennes": {
        metric: {
            "mean": round(moyennes_metriques[metric], 4),
            "std": round(stds_metriques[metric], 4)
        } for metric in moyennes_metriques
    },
    "details_par_fold": scores_folds
}

# 4. Sauvegarde
filename = "results_mlp_TF-ID.json"
with open(filename, 'w') as f:
    json.dump(resultats_complets, f, indent=4)

print(f"✅ Rapport MLP sauvegardé : {filename}")

# Affichage console pour comparaison immédiate
print(f"⏱️ Temps MLP : {temps_moyen:.3f}s (± {temps_std:.3f}s)")
print(f"🎯 F1-Score MLP : {resultats_complets['performances_moyennes']['F1_Score']['mean']} (± {resultats_complets['performances_moyennes']['F1_Score']['std']})")

--- Début de la Validation Croisée sur cpu ---

🚀 Fold 1/5
Accuracy: 0.8887| F1-Score: 0.9380 | ROC-AUC: 0.8444

🚀 Fold 2/5
Accuracy: 0.8862| F1-Score: 0.9362 | ROC-AUC: 0.8383

🚀 Fold 3/5
Accuracy: 0.8930| F1-Score: 0.9403 | ROC-AUC: 0.8450

🚀 Fold 4/5
Accuracy: 0.8966| F1-Score: 0.9424 | ROC-AUC: 0.8564

🚀 Fold 5/5
Accuracy: 0.8910| F1-Score: 0.9388 | ROC-AUC: 0.8592
✅ Rapport MLP sauvegardé : results_mlp_TF-ID.json
⏱️ Temps MLP : 5.014s (± 1.954s)
🎯 F1-Score MLP : 0.9392 (± 0.0023)


In [4]:
# --- Configuration ---
VECTOR_SIZE = 300  # Dimension de l'espace sémantique
K_FOLDS = 5
X = df_train['Texte']
y = np.array(df_train['Cible_Chirac'])

# 1. Préparation des données (Word2Vec a besoin de listes de mots)
# On tokenize une seule fois pour gagner du temps
X_tokenized = X.apply(lambda x: str(x).split())

def get_document_vector(tokens, model, vector_size):
    """Calcule la moyenne des vecteurs des mots présents dans le dictionnaire."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# --- Initialisation ---
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
scores_folds = []
durees_folds = []

print(f"--- Début de la Validation Croisée (Word2Vec) ---")

temps_debut_total = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(X_tokenized, y)):
    print(f"\n🚀 Fold {fold + 1}/{K_FOLDS}")
    # Séparation et Vectorisation

    # Séparation
    X_train_tokens = X_tokenized.iloc[train_idx]
    X_val_tokens = X_tokenized.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # 2. Entraînement de Word2Vec (uniquement sur le train du fold)
    w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=VECTOR_SIZE, window=5, min_count=2,workers=4)

    # 3. Vectorisation des documents (Moyenne des vecteurs)
    X_train_w2v = np.array([get_document_vector(text, w2v_model, VECTOR_SIZE) for text in X_train_tokens])
    X_val_w2v = np.array([get_document_vector(text, w2v_model, VECTOR_SIZE) for text in X_val_tokens])

    # 4. Modèle (Ici LogReg pour l'exemple, adaptable au MLP)
    modele = LogisticRegression(max_iter=5000, C=1.0)
    start_fold = time.time()
    modele.fit(X_train_w2v, y_train)

    # Inférence
    probs = modele.predict_proba(X_val_w2v)[:, 1]
    preds = modele.predict(X_val_w2v)

    # Métriques
    metriques = {
        'Fold': fold + 1,
        'Accuracy': accuracy_score(y_val, preds),
        'Precision': precision_score(y_val, preds),
        'Recall': recall_score(y_val, preds),
        'F1_Score': f1_score(y_val, preds),
        'ROC_AUC': roc_auc_score(y_val, probs)
    }
    
    scores_folds.append(metriques)
    durees_folds.append(time.time() - start_fold)

# --- 5. Consolidation et Sauvegarde JSON ---
temps_total = time.time() - temps_debut_total
df_results = pd.DataFrame(scores_folds).drop(columns=['Fold'])

resultats_complets = {
    "config": {
        "nom_modele": "Logistic_Regression_W2V",
        "representation": "Word2Vec",
        "nb_parametres_totaux": int((VECTOR_SIZE * 1) + 1),
        "taille_vecteur_entree": VECTOR_SIZE,
        "nb_folds": K_FOLDS,
        "device_utilise": "cpu"
    },
    "temps_execution": {
        "total_sec": round(temps_total, 2),
        "moyen_par_fold_sec": round(np.mean(durees_folds), 4),
        "std_par_fold_sec": round(np.std(durees_folds), 4)
    },
    "performances_moyennes": {
        metric: {
            "mean": round(df_results[metric].mean(), 4),
            "std": round(df_results[metric].std(), 4)
        } for metric in df_results.columns
    }
}

with open('results_word2vec_logreg.json', 'w') as f:
    json.dump(resultats_complets, f, indent=4)

print("✅ Rapport Word2Vec sauvegardé.")

--- Début de la Validation Croisée (Word2Vec) ---

🚀 Fold 1/5

🚀 Fold 2/5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'



🚀 Fold 3/5

🚀 Fold 4/5

🚀 Fold 5/5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


✅ Rapport Word2Vec sauvegardé.


In [4]:
from gensim.models import KeyedVectors

# Utiliser pour crée les embedding mais mtn suprrimer

#ft = KeyedVectors.load_word2vec_format("cc.fr.300.vec", binary=False)
#print(ft.most_similar("politique"))
#print(ft["chirac"].shape)  # (300,)

In [6]:
# ── Utilisation ───────────────────────────────────────────────────────────────
df_train = load_and_prepare(
    corpus_path = "Data/train/corpus.tache1.learn.utf8",
    pkl_path    = "sequences_fasttext_fr.pkl",
    vector_size = 300,
)

# Vérification rapide
print(df_train.columns.tolist())
print(df_train.head(3))

📂 Chargement du corpus...
   → 57413 lignes chargées
🧹 Nettoyage du texte...
📦 Chargement des séquences depuis 'sequences_fasttext_fr.pkl'...
   → 57413 séquences fusionnées
🗑️  Suppression des lignes vides...
   → 35 lignes supprimées | 57378 lignes restantes
🔍 Vérification cohérence tokens ↔ vecteurs...


Vérification: 100%|██████████| 57378/57378 [00:01<00:00, 39686.78it/s]



✅ Lignes finales       : 57378
✅ Balance C/M         : {'C': 49855, 'M': 7523}
✅ Erreurs restantes  : 0
🎉 Pipeline prêt pour l'entraînement !
['Doc_ID', 'Sentence_ID', 'Label', 'Texte', 'Cible_Chirac', 'Texte_Nettoye', 'Sequence']
   Doc_ID  Sentence_ID Label  \
0     100            1     C   
1     100            2     C   
2     100            3     C   

                                               Texte  Cible_Chirac  \
0  Quand je dis chers amis, il ne s'agit pas là d...             1   
1  D'abord merci de cet exceptionnel accueil que ...             1   
2  C'est toujours très émouvant de venir en Afriq...             1   

                                       Texte_Nettoye  \
0  quand dis chers amis agit là formule diplomati...   
1  abord merci cet exceptionnel accueil congolais...   
2  toujours très émouvant venir afrique car proba...   

                                            Sequence  
0  [[0.0745, -0.0541, 0.004, 0.0394, -0.1497, -0....  
1  [[-0.0682, 0.0843, 0

In [ ]:
# --- Configuration ---
VECTOR_SIZE = 300
K_FOLDS = 5
X = df_train['sequence']
y = np.array(df_train['Cible_Chirac'])

# 1. Préparation des données (Word2Vec a besoin de listes de mots)
# On tokenize une seule fois pour gagner du temps
X_tokenized = X.apply(lambda x: str(x).split())

def get_document_vector(tokens, model, vector_size):
    """Calcule la moyenne des vecteurs des mots présents dans le dictionnaire."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# --- Initialisation ---
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
scores_folds = []
durees_folds = []

print(f"--- Début de la Validation Croisée (Word2Vec) ---")

temps_debut_total = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(X_tokenized, y)):
    print(f"\n🚀 Fold {fold + 1}/{K_FOLDS}")
    # Séparation et Vectorisation

    # Séparation
    X_train_tokens = X_tokenized.iloc[train_idx]
    X_val_tokens = X_tokenized.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # 3. Vectorisation des documents (Moyenne des vecteurs)
    X_train_w2v = np.array([get_document_vector(text, w2v, VECTOR_SIZE) for text in X_train_tokens])
    X_val_w2v = np.array([get_document_vector(text, w2v, VECTOR_SIZE) for text in X_val_tokens])

    # 4. Modèle (Ici LogReg pour l'exemple, adaptable au MLP)
    modele = LogisticRegression(max_iter=5000, C=1.0)
    start_fold = time.time()
    modele.fit(X_train_w2v, y_train)

    # Inférence
    probs = modele.predict_proba(X_val_w2v)[:, 1]
    preds = modele.predict(X_val_w2v)

    # Métriques
    metriques = {
        'Fold': fold + 1,
        'Accuracy': accuracy_score(y_val, preds),
        'Precision': precision_score(y_val, preds),
        'Recall': recall_score(y_val, preds),
        'F1_Score': f1_score(y_val, preds),
        'ROC_AUC': roc_auc_score(y_val, probs)
    }
    
    scores_folds.append(metriques)
    durees_folds.append(time.time() - start_fold)

# --- 5. Consolidation et Sauvegarde JSON ---
temps_total = time.time() - temps_debut_total
df_results = pd.DataFrame(scores_folds).drop(columns=['Fold'])

resultats_complets = {
    "config": {
        "nom_modele": "Logistic_Regression_W2V",
        "representation": "Word2Vec",
        "nb_parametres_totaux": int((VECTOR_SIZE * 1) + 1),
        "taille_vecteur_entree": VECTOR_SIZE,
        "nb_folds": K_FOLDS,
        "device_utilise": "cpu"
    },
    "temps_execution": {
        "total_sec": round(temps_total, 2),
        "moyen_par_fold_sec": round(np.mean(durees_folds), 4),
        "std_par_fold_sec": round(np.std(durees_folds), 4)
    },
    "performances_moyennes": {
        metric: {
            "mean": round(df_results[metric].mean(), 4),
            "std": round(df_results[metric].std(), 4)
        } for metric in df_results.columns
    }
}

with open('results_word2vec_logreg.json', 'w') as f:
    json.dump(resultats_complets, f, indent=4)

print("✅ Rapport Word2Vec sauvegardé.")